# 대표 장르 추출

## 목적
Steam 인디 게임 9,692개 중 리뷰 감성 분석을 위한 샘플링 대상 게임을 선별하기 위해, 각 게임의 대표 장르를 추출한다.

## 배경
- Steam `appdetails` API의 `genres` 필드는 알파벳 순으로 정렬된 멀티레이블 리스트로, "주 장르" 개념이 없음
- 층화 샘플링을 위해 게임당 하나의 대표 장르 배정이 필요
- 대표 장르 선정 방식: **희귀 장르 우선** — 게임이 가진 장르 중 전체 데이터에서 등장 빈도가 가장 낮은 장르를 대표로 선정

## 한계
- 희귀 장르가 반드시 해당 게임의 핵심 장르를 의미하지는 않음
- Sports/Racing 버킷에는 해당 장르가 부수적인 게임이 포함될 수 있음
- Steam 데이터 특성상 완전한 대표 장르 선정은 불가능하며, 이 한계를 감안하여 해석 필요

## 라이브러리 임포트

In [1]:
import ast
import warnings
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

pd.set_option('display.max_columns', None)
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings('ignore')

print('라이브러리 로드 완료')

라이브러리 로드 완료


## 데이터 로드 및 파생 컬럼 생성

`steam_indie_9692.csv`를 불러오고 분석에 필요한 파생 컬럼을 추가합니다.

- `total_reviews`: 긍정 + 부정 리뷰 합산
- `positive_rate`: 긍정 리뷰 비율 (%)
- `genres`: 문자열 리스트 파싱

In [2]:
df = pd.read_csv('../../../data/processed/steam_indie_9692.csv')

df['total_reviews'] = df['positive'] + df['negative']
df['release_date']  = pd.to_datetime(df['release_date'], errors='coerce')
df['positive_rate'] = df['positive'] / df['total_reviews'] * 100

def parse_genres(g):
    try:
        return ast.literal_eval(g)
    except Exception:
        return []

df['genres'] = df['genres'].apply(parse_genres)

print(f'모집단: {len(df):,}개')
print(f'출시연도 분포:')
print(df['release_date'].dt.year.value_counts().sort_index())

모집단: 9,692개
출시연도 분포:
release_date
2023    3498
2024    4180
2025    2014
Name: count, dtype: int64


## 1. 전체 장르 현황 탐색

분석 대상 장르를 결정하기 위해 전체 장르의 게임 수 분포를 확인합니다.

In [3]:
genre_counts_raw = {}
for genres in df['genres']:
    for genre in genres:
        genre_counts_raw[genre] = genre_counts_raw.get(genre, 0) + 1

genre_df = pd.DataFrame(list(genre_counts_raw.items()), columns=['Genre', 'Count']).sort_values('Count', ascending=False)
display(genre_df)

,Genre,Count
2,Indie,9687
1,Adventure,4807
7,Casual,4111
0,Action,4093
4,Simulation,2503
3,RPG,2194
5,Strategy,2005
9,Sports,341
8,Racing,301
6,Massively Multiplayer,124


In [4]:
median_count = genre_df['Count'].median()
mean_count   = genre_df['Count'].mean()

print(f'--- 장르별 게임 수 통계 ---')
print(f'중앙값: {median_count:.1f}개')
print(f'평균:   {mean_count:.1f}개')
print(f'표준편차: {genre_df["Count"].std():.1f}개')
print(f'최소값: {genre_df["Count"].min()}개')
print(f'최대값: {genre_df["Count"].max()}개')

above_median = genre_df[genre_df['Count'] >= median_count]
below_median = genre_df[genre_df['Count'] <  median_count]

print(f'\n=== 중앙값 이상 장르 ({len(above_median)}개) ===')
print(above_median[['Genre', 'Count']].sort_values('Count', ascending=False).to_string(index=False))
print(f'\n=== 중앙값 미만 장르 ({len(below_median)}개) ===')
print(below_median[['Genre', 'Count']].sort_values('Count', ascending=False).to_string(index=False))

--- 장르별 게임 수 통계 ---
중앙값: 21.0개
평균:   1439.7개
표준편차: 2473.8개
최소값: 1개
최대값: 9687개

=== 중앙값 이상 장르 (11개) ===
                Genre  Count
                Indie   9687
            Adventure   4807
               Casual   4111
               Action   4093
           Simulation   2503
                  RPG   2194
             Strategy   2005
               Sports    341
               Racing    301
Massively Multiplayer    124
            Utilities     21

=== 중앙값 미만 장르 (10개) ===
                Genre  Count
Design & Illustration     10
 Animation & Modeling      7
            Education      6
    Software Training      6
     Game Development      5
     Video Production      4
     Audio Production      4
        Photo Editing      2
       Web Publishing      2
           Accounting      1


## 2. 대상 장르별 게임 수 분포

8개 대상 장르(`TARGET_GENRES`) 각각에 몇 개의 게임이 속하는지 확인합니다.
한 게임이 여러 장르를 가질 수 있으므로 장르별 집계는 중복을 허용합니다.

In [5]:
TARGET_GENRES = {'Adventure', 'Casual', 'Action', 'Simulation', 'RPG', 'Strategy', 'Sports', 'Racing'}

# 대상 장르만 필터링하여 게임 수 집계
target_genre_counts = {
    genre: sum(1 for genres in df['genres'] if genre in genres)
    for genre in TARGET_GENRES
}
target_genre_df = (
    pd.DataFrame(list(target_genre_counts.items()), columns=['genre', 'count'])
    .sort_values('count', ascending=False)
    .reset_index(drop=True)
)

fig = px.bar(
    target_genre_df,
    x='genre',
    y='count',
    text='count',
    title='대상 장르별 게임 수 분포 (중복 허용)',
    labels={'genre': '장르', 'count': '게임 수'},
    color='count',
    color_continuous_scale='Blues',
)
fig.update_traces(textposition='outside')
fig.update_layout(
    xaxis_categoryorder='total descending',
    coloraxis_showscale=False,
    height=450,
)
fig.show()

print(target_genre_df.to_string(index=False))

     genre  count
 Adventure   4807
    Casual   4111
    Action   4093
Simulation   2503
       RPG   2194
  Strategy   2005
    Sports    341
    Racing    301


## 3. 대상 장르 선정 및 대표 장르 추출

분석 대상 장르는 게임 수가 100개 이상이며 실제 게임 장르에 해당하는 8개로 한정한다.

**제외 기준**

| 제외 장르 | 이유 |
|---|---|
| Indie | 거의 모든 게임에 포함되어 변별력 없음 |
| Massively Multiplayer | 게임 수 부족 |
| Design & Illustration, Animation & Modeling, Education, Software Training, Game Development, Video Production, Audio Production, Photo Editing, Web Publishing, Accounting | 장르별 게임 수 중앙값(21개) 미만으로 통계적 분석에 부적합 |
| Utilities | 도구 성 게임 제외 |
   
| 최종 대상 장르 |
|----------|
| Action, Adventure, Casual, Simulation, RPG, Strategy, Sports, Racing |

In [6]:
# 8개 대상 장르만 필터링
df['genres_filtered'] = df['genres'].apply(lambda g: [x for x in g if x in TARGET_GENRES])

# 대상 장르가 하나도 없는 게임 제외
df_filtered = df[df['genres_filtered'].map(len) > 0].copy()

# 8개 장르 내 희귀도 계산
genre_count = Counter(genre for genres in df_filtered['genres_filtered'] for genre in genres)
print('=== 장르별 게임 수 (희귀도 기준) ===')
for genre, count in sorted(genre_count.items(), key=lambda x: x[1]):
    print(f'  {genre:25s}: {count:,}')

# 희귀 장르 우선으로 대표 장르 선정
df_filtered['primary_genre'] = df_filtered['genres_filtered'].apply(
    lambda genres: min(genres, key=lambda g: genre_count[g])
)

print('\n=== 대표 장르 분포 ===')
print(df_filtered['primary_genre'].value_counts().to_string())

=== 장르별 게임 수 (희귀도 기준) ===
  Racing                   : 301
  Sports                   : 341
  Strategy                 : 2,005
  RPG                      : 2,194
  Simulation               : 2,503
  Action                   : 4,093
  Casual                   : 4,111
  Adventure                : 4,807

=== 대표 장르 분포 ===
primary_genre
Action        2282
Strategy      1912
RPG           1537
Simulation    1188
Casual        1091
Adventure      759
Racing         301
Sports         244


## 4. 결과 확인

게임별 필터링된 장르 목록과 선정된 대표 장르를 확인한다.

In [7]:
df_filtered[['appid', 'name', 'genres_filtered', 'primary_genre']].head(20)

,appid,name,genres_filtered,primary_genre
0,899770,Last Epoch,"[Action, Adventure, RPG]",RPG
1,251570,7 Days to Die,"[Action, Adventure, RPG, Simulation, Strategy]",Strategy
2,1116170,CyberCorp,"[Action, Adventure, RPG]",RPG
3,1326470,Sons Of The Forest,"[Action, Adventure, Simulation]",Simulation
4,2186680,"Warhammer 40,000: Rogue Trader","[Action, Adventure, RPG, Strategy]",Strategy
5,526870,Satisfactory,"[Adventure, Simulation, Strategy]",Strategy
6,2881650,Content Warning,"[Action, Adventure]",Action
7,513710,SCUM,"[Action, Adventure]",Action
8,1144200,Ready or Not,"[Action, Adventure]",Action
9,1145350,Hades II,"[Action, RPG]",RPG


In [8]:
genre_dist = df_filtered['primary_genre'].value_counts().reset_index()
genre_dist.columns = ['genre', 'count']

fig = px.bar(
    genre_dist,
    x='genre',
    y='count',
    text='count',
    title='대표 장르별 게임 수 분포',
    labels={'genre': '장르', 'count': '게임 수'},
)
fig.update_traces(textposition='outside')
fig.update_layout(xaxis_categoryorder='total descending')
fig.show()